# Module 1 — CPU solutions

No model download, GPU or paid service is needed. These are formative exercises with explicit fixtures. They do not count as measured capstone evidence. Run all cells in order; assertions check the worked results.

## 1. Token probabilities

Implement stable softmax. Check that adding a large constant to every logit does not change probabilities.

In [ ]:
import math
def softmax(logits):
    if not logits or any(not math.isfinite(x) for x in logits): raise ValueError("Finite nonempty logits required")
    weights=[math.exp(x-max(logits)) for x in logits]
    return [x/sum(weights) for x in weights]
a=softmax([1.,2.,3.]);b=softmax([10001.,10002.,10003.])
assert all(abs(x-y)<1e-12 for x,y in zip(a,b))
assert abs(sum(a)-1)<1e-12
print(a)


## 2. Failures belong in metrics

Compute accuracy, defect F1 and a confusion table including invalid replies. Use 2 defective and 2 good cases, with one defective reply invalid.

In [ ]:
rows=[("defective","defective"),("defective","invalid"),("good","good"),("good","invalid")]
tp=sum(t==p=="defective" for t,p in rows)
fp=sum(t=="good" and p=="defective" for t,p in rows)
fn=sum(t=="defective" and p!="defective" for t,p in rows)
accuracy=sum(t==p for t,p in rows)/len(rows)
f1=2*tp/(2*tp+fp+fn)
matrix=[[sum(t==truth and p==pred for t,p in rows) for pred in ["good","defective","invalid"]] for truth in ["good","defective"]]
assert accuracy==.5 and abs(f1-2/3)<1e-12 and sum(map(sum,matrix))==4
print({"accuracy":accuracy,"f1":f1,"confusion":matrix})


## 3. Freeze before inspecting

Build a deterministic stratified split from 60 unique identities: 10 development and 20 test per class. Detect overlap and duplicate identities.

In [ ]:
import random,hashlib,json
rows=[{"id":f"{label}-{i}","label":label} for label in ["good","defective"] for i in range(30)]
dev=[];test=[]
for label in ["good","defective"]:
    group=[r for r in rows if r["label"]==label];random.Random(42).shuffle(group)
    dev+=group[:10];test+=group[10:]
assert len(test)==40 and len(dev)==20
assert not {r["id"] for r in test}&{r["id"] for r in dev}
assert all(sum(r["label"]==label for r in test)==20 for label in ["good","defective"])
print(hashlib.sha256(json.dumps({"dev":dev,"test":test},sort_keys=True).encode()).hexdigest())


## 4. Strict output parsing

Accept only the expected inspection schema. Preserve malformed output as PARSE_ERROR; never turn it into GOOD.

In [ ]:
import json
def parse(raw):
    try:
        r=json.loads(raw)
        assert set(r)=={"verdict","defect_type","description"}
        assert r["verdict"] in {"GOOD","DEFECTIVE"} and isinstance(r["description"],str) and r["description"].strip()
        assert r["defect_type"] is None or isinstance(r["defect_type"],str)
        assert r["verdict"]!="DEFECTIVE" or r["defect_type"]
        return r
    except (ValueError,TypeError,AssertionError):return {"verdict":"PARSE_ERROR","raw":raw}
assert parse("not json")["verdict"]=="PARSE_ERROR"
assert parse('{"verdict":"GOOD","defect_type":null,"description":"No visible crack."}')["verdict"]=="GOOD"
print(parse("not json"))


## Transfer to the model lab

Open `lab.ipynb`. Identify the corresponding real-data function, record the difference between fixture and model evidence, and explain one failure that a numerical score alone would miss.